### `Join` en Spark

Ahora queremos combinar los dos _DataFrames_ que creamos durante la [sesión anterior](./groupby_en_spark.ipynb) en una única tabla y enriquecerla con el nombre de cada zona.

In [3]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('groupby-en-spark') \
    .getOrCreate()

#### Unir los ingresos de taxis verdes y amarillos

Antes de unir los dos _DataFrames_, renombramos sus columnas para distinguir el origen de cada métrica.

In [6]:
df_green_revenue = spark.read.parquet('/data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('/data/report/revenue/yellow')

df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

Leemos los resultados que materializamos en el artículo anterior y renombramos las columnas `amount` y `number_records` de cada _DataFrame_. Sin este paso, al unirlos tendríamos dos columnas llamadas `amount` y no sabríamos cuál es de cada tipo de taxi.

In [11]:
df_join = df_green_revenue_tmp.join(
    df_yellow_revenue_tmp,
    on=['hour', 'zone'],
    how='outer'
)

df_join.show()
df_join.write.parquet('/data/report/revenue/total', mode='overwrite')

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   3|              NULL|                NULL|              25.0|                    1|
|2020-01-01 00:00:00|   4|              NULL|                NULL|1004.3000000000002|                   57|
|2020-01-01 00:00:00|  40|168.98000000000002|                   8|             89.97|                    5|
|2020-01-01 00:00:00|  45|              NULL|                NULL| 732.4800000000002|                   42|
|2020-01-01 00:00:00|  47|              13.3|                   1|               8.3|                    1|
|2020-01-01 00:00:00|  51|              17.8|                   2|              31.0|                    1|
|2020-01-01 00:00:00|  62|  

26/03/07 16:07:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

* Usamos `how='outer'` porque queremos conservar todas las combinaciones de hora y zona, independientemente de si aparecen en los datos de taxis verdes, en los de amarillos o en ambos. Si una combinación solo existe en un lado, el otro tendrá `null`.
* Con `how='inner'` perderíamos todas las filas que no tienen pareja en el otro _DataFrame_.

#### Enriquecer con datos de zonas

El segundo tipo de _join_ se produce cuando uno de los dos _DataFrames_ es muy pequeño. Queremos añadir el nombre de cada zona a nuestra tabla de ingresos:

In [10]:
df_join = spark.read.parquet('/data/report/revenue/total')
df_zones = spark.read.csv('/data/homework/raw/taxi_zone_lookup.csv', header=True)

df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)
df_result.drop('LocationID', 'zone').write.parquet('/data/revenue-zones')

26/03/07 16:03:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Al ejecutar este trabajo, la interfaz de Spark muestra algo llamativo: **solo hay una etapa**, no dos. Esto se debe a que Spark detecta automáticamente que `df_zones` es una tabla muy pequeña y aplica una estrategia diferente: el _broadcast join_.